# Дообучение Qwen3.5 для проверки чатов (CQA)

Ноутбук учит модель ставить оценки по чек-листу Freedom так, как в эталонных разборах из `cqa-synthetic`.

**Перед запуском**
1. *Среда выполнения → Сменить среду выполнения → GPU.*
   - **L4 или A100** (Colab Pro): обучится **Qwen3.5-9B**, это та же модель, что скачана на ноутбук. Ей нужно около 22 ГБ видеопамяти.
   - **T4** (бесплатно): обучится **Qwen3.5-4B**, около 10 ГБ. 9B на T4 не помещается, а обучение в 4 битах Unsloth для Qwen3.5 не рекомендует.
2. Запускайте ячейки по порядку. Данные уже встроены во 2-ю ячейку, загружать ничего не нужно.
3. Если после установки Colab попросит перезапустить среду, перезапустите и продолжите со 2-й ячейки.

В конце получится файл `.gguf`, который запускается через llama.cpp так же, как скачанная модель.

In [ ]:
# 1. Установка. Qwen3.5 требует transformers 5.
!pip install -q --upgrade unsloth unsloth_zoo
!pip install -q --upgrade "transformers>=5.0.0"
import transformers; print('transformers', transformers.__version__)

In [ ]:
# 2. Данные встроены в ноутбук (gzip + base64), загружать файлы не нужно
import base64, gzip
DATA = {
    'train.jsonl': '...',
    'val.jsonl': '...',
}
for name, blob in DATA.items():
    open(name, 'wb').write(gzip.decompress(base64.b64decode(blob)))
from datasets import load_dataset
data = load_dataset('json', data_files={'train': 'train.jsonl', 'val': 'val.jsonl'})
print(data)

In [ ]:
# 3. Выбор модели по объёму видеопамяти
import torch
assert torch.cuda.is_available(), 'Включите GPU: Среда выполнения → Сменить среду выполнения'
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
MODEL = 'unsloth/Qwen3.5-9B' if vram >= 22 else 'unsloth/Qwen3.5-4B'
print(f'{torch.cuda.get_device_name(0)}: {vram:.1f} ГБ → {MODEL}')
MAX_LEN = 4096  # самый длинный пример ≈ 7000 символов, это меньше 3000 токенов

In [ ]:
# 4. Загрузка модели и LoRA (16 бит, как рекомендует Unsloth для Qwen3.5)
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL,
    max_seq_length = MAX_LEN,
    load_in_4bit = False,
    load_in_16bit = True,
    full_finetuning = False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
    max_seq_length = MAX_LEN,
)

In [ ]:
# 5. Превращаем диалоги в текст по шаблону чата модели (без режима рассуждений)
text_tok = getattr(tokenizer, 'tokenizer', tokenizer)
def to_text(batch):
    return {'text': [text_tok.apply_chat_template(m, tokenize=False, enable_thinking=False) for m in batch['messages']]}
train = data['train'].map(to_text, batched=True)
val = data['val'].map(to_text, batched=True)
lengths = [len(text_tok(t).input_ids) for t in train['text']]
print('Токенов в примере: макс', max(lengths), 'среднее', sum(lengths)//len(lengths))
print(train[0]['text'][-400:])

In [ ]:
# 6. Обучение. Учим только ответ модели (оценку), а не системную инструкцию и диалог.
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only
trainer = SFTTrainer(
    model = model,
    tokenizer = text_tok,
    train_dataset = train,
    eval_dataset = val,
    args = SFTConfig(
        dataset_text_field = 'text',
        max_seq_length = MAX_LEN,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_train_epochs = 2,
        learning_rate = 2e-4,
        warmup_steps = 10,
        lr_scheduler_type = 'linear',
        optim = 'adamw_8bit',
        weight_decay = 0.01,
        logging_steps = 5,
        eval_strategy = 'epoch',
        save_strategy = 'no',
        seed = 3407,
        output_dir = 'outputs',
        report_to = 'none',
    ),
)
trainer = train_on_responses_only(trainer, instruction_part='<|im_start|>user\n', response_part='<|im_start|>assistant\n')
stats = trainer.train()
print(stats)

In [ ]:
# 7. Проверка на 26 диалогах, которых модель не видела при обучении
import json, re
FastLanguageModel.for_inference(model)
IDS = ['contact', 'needs', 'identity', 'consultation', 'empathy', 'management', 'closing', 'gross']
hits = {k: 0 for k in IDS}; broken = 0
for row in data['val']:
    prompt = text_tok.apply_chat_template(row['messages'][:2], tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = text_tok(prompt, return_tensors='pt').to('cuda')
    out = model.generate(**inputs, max_new_tokens=1200, do_sample=False)
    answer = text_tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    expected = json.loads(row['messages'][2]['content'])['evaluations']
    try:
        got = json.loads(re.search(r'\{.*\}', answer, re.S).group(0))['evaluations']
    except Exception:
        broken += 1; print(row['id'], 'не JSON:', answer[:200]); continue
    for k in IDS:
        hits[k] += got.get(k, {}).get('value') == expected[k]['value']
n = len(data['val'])
print(f'Сломанных ответов: {broken} из {n}')
for k in IDS: print(f'{k:13} совпало {hits[k]}/{n}')
print(f'Всего совпало: {sum(hits.values())}/{n*len(IDS)}')

In [ ]:
# 8. Сохранение: адаптер LoRA и готовый файл GGUF (q4_k_m) для llama.cpp
name = 'cqa-' + MODEL.split('/')[-1].lower()
model.save_pretrained(name + '-lora'); text_tok.save_pretrained(name + '-lora')
model.save_pretrained_gguf(name + '-gguf', tokenizer, quantization_method='q4_k_m')
!ls -la {name}-gguf

In [ ]:
# 9. Скачать результат. Файл большой (3–6 ГБ), поэтому надёжнее через Google Drive.
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/cqa-model && cp -r {name}-gguf {name}-lora /content/drive/MyDrive/cqa-model/
print('Готово: Google Drive → cqa-model')

## Запуск обученной модели на ноутбуке

Скачайте `.gguf` из Google Drive в `D:\models` и запустите:

```
llama-server -m D:\models\<файл>.gguf -c 4096 -t 8 -ngl 0 --device none --port 8080 --reasoning-budget 0
```

Проверить качество на ноутбуке: `node eval_local.mjs` в папке `cqa-training`.